# Generative J-cone validation on Gemma 4 E4B — Colab runner

Reruns the corrected two-concept generative steering experiment with **visible
monitoring** and **persistent, Drive-backed outputs**, so a terminated VM can no
longer take a completed run with it.

Sections:

1. GPU / runtime verification
2. Repository setup (clone or update `experiment/generative-jlens-validation`)
3. Hugging Face authentication (`getpass`; the token is never printed or written)
4. Google Drive mount
5. Lens restoration and checksum verification
6. Target-token validation
7. The corrected two-concept experiment (writes directly to Drive)
8. Live GPU / log / progress monitoring
9. Result and provenance inspection
10. Persistent archive and export

**Why the provenance section exists.** In the lost run
`generative_20260730T164407987257_0c5fc4c2f7a5`, several `dev-entity-mandela`
records decoded `Black Hole` — the surface form of `held-phrase-black-hole`,
another benchmark target. The code audit found no example-indexing bug, and that
run's config fingerprint pins it to `split: dev`, which cannot reach a held-out
example's cone. But the records of the time could not *prove* it either way:
nothing in them said which example each activation and cone actually came from.

Records now carry that identity block (`source_example_id`,
`cone_source_example_id`, `donor_example_id`, activation and cone norms plus
sha256 fingerprints), and section 9 checks it directly. Section 9 also prints
the `none` (unsteered) decode per receiver prompt — the receiver prompt carries
no example information, so if the repeated output is simply what the model says
to that prompt, the baseline row will show it.

**Every cell is safe to rerun.** Nothing here deletes a previous run.

## 1. GPU and runtime verification

No model load, no downloads. Fails loudly if there is no CUDA device, since the
experiment needs one.

In [ ]:
# 1. Runtime facts: Python, PyTorch, Transformers, CUDA, GPU, memory, nvidia-smi.
import platform
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"IN_COLAB          = {IN_COLAB}")
print(f"Python            = {platform.python_version()}  ({sys.executable})")

try:
    import torch
except ImportError:
    torch = None
    print("PyTorch           = NOT INSTALLED (section 2 installs it)")
else:
    print(f"PyTorch           = {torch.__version__}")
    print(f"CUDA (torch)      = {torch.version.cuda}")
    print(f"cuda.is_available = {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        print(f"GPU               = {properties.name}")
        print(f"GPU memory        = {properties.total_memory / 1024**3:.1f} GiB")
        free, total = torch.cuda.mem_get_info()
        print(f"GPU free / total  = {free / 1024**3:.1f} / {total / 1024**3:.1f} GiB")

try:
    import transformers
except ImportError:
    print("Transformers      = NOT INSTALLED (section 2 installs it)")
else:
    print(f"Transformers      = {transformers.__version__}")

if shutil.which("nvidia-smi"):
    print("\n" + subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    ).stdout)
else:
    print("\nnvidia-smi not on PATH.")

HAS_CUDA = bool(torch is not None and torch.cuda.is_available())
if IN_COLAB and not HAS_CUDA:
    raise RuntimeError(
        "No CUDA device. Runtime > Change runtime type > GPU (L4 or better), "
        "then rerun from section 1. The experiment loads Gemma 4 E4B and will "
        "not fit on CPU."
    )

## 2. Repository setup

Clones (or updates) `experiment/generative-jlens-validation`, asserts the
checked-out HEAD, and installs the package with `pip install -e .`.

The HEAD assertion has two halves:

- **Structural**: the checkout is on the expected branch and matches
  `origin/<branch>`, i.e. the runtime is not sitting on a stale local commit.
- **Semantic**: `jlens.generative` exposes the per-example provenance API this
  notebook's section 9 depends on. A checkout that predates the provenance
  commit fails here rather than producing an unreadable run.

Set `EXPECTED_HEAD` to a specific 40-character commit to pin an exact revision.

In [ ]:
# 2. Clone or update the branch, assert HEAD, and install the package.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
BRANCH = "experiment/generative-jlens-validation"
CHECKOUT_DIR = Path("/content/jacobian-lens-gemma") if IN_COLAB else Path.cwd()
EXPECTED_HEAD = None  # set to a full 40-char sha to pin an exact revision


def git(*args, cwd=CHECKOUT_DIR, check=True):
    result = subprocess.run(
        ["git", *args], cwd=str(cwd), capture_output=True, text=True, check=False
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed ({result.returncode}):\n"
            f"{result.stdout}\n{result.stderr}"
        )
    return result.stdout.strip()


def _auth_url():
    """Repo URL with a token only if one is available; never printed."""
    if not IN_COLAB:
        return REPO_URL
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        return REPO_URL  # public clone; fine if the repo is public
    return REPO_URL.replace("https://", f"https://x-access-token:{token}@")


if IN_COLAB:
    if not (CHECKOUT_DIR / ".git").is_dir():
        CHECKOUT_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, _auth_url(), str(CHECKOUT_DIR)],
            check=True,
            capture_output=True,
            text=True,
        )
        print(f"cloned {REPO_URL} @ {BRANCH}")
    else:
        print(f"existing checkout at {CHECKOUT_DIR} — updating")
    # Keep the token out of .git/config: fetch with a per-invocation URL.
    git("fetch", "--quiet", _auth_url(), BRANCH)
    git("checkout", "--quiet", "-B", BRANCH, "FETCH_HEAD")

HEAD = git("rev-parse", "HEAD")
BRANCH_NAME = git("rev-parse", "--abbrev-ref", "HEAD")
print(f"checkout : {CHECKOUT_DIR}")
print(f"branch   : {BRANCH_NAME}")
print(f"HEAD     : {HEAD}")
print(f"subject  : {git('log', '-1', '--pretty=%s')}")

if EXPECTED_HEAD and HEAD != EXPECTED_HEAD:
    raise RuntimeError(
        f"HEAD assertion failed: checkout is at {HEAD}, EXPECTED_HEAD is "
        f"{EXPECTED_HEAD}. Refusing to run against an unexpected revision."
    )
if IN_COLAB:
    if BRANCH_NAME != BRANCH:
        raise RuntimeError(
            f"HEAD assertion failed: on branch {BRANCH_NAME!r}, expected {BRANCH!r}."
        )
    remote_head = git("rev-parse", "FETCH_HEAD")
    if HEAD != remote_head:
        raise RuntimeError(
            f"HEAD assertion failed: HEAD {HEAD} != origin/{BRANCH} {remote_head}. "
            f"Rerun this cell to update the checkout."
        )
    print(f"HEAD matches origin/{BRANCH}")

os.chdir(CHECKOUT_DIR)
if str(CHECKOUT_DIR) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_DIR))

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=str(CHECKOUT_DIR),
    capture_output=True,
    text=True,
    check=False,
)
print(install.stdout[-2000:])
if install.returncode != 0:
    print(install.stderr[-4000:])
    raise RuntimeError(f"pip install -e . failed ({install.returncode})")
print("pip install -e . OK")

In [ ]:
# 2b. Semantic HEAD assertion: the provenance API section 9 reads must exist.
import importlib

import jlens.generative as jgen

importlib.reload(jgen)
REQUIRED_API = (
    "cone_source_role",
    "expected_cone_source_example_id",
    "tensor_sha256",
    "vector_identity",
)
missing = [name for name in REQUIRED_API if not hasattr(jgen, name)]
if missing:
    raise RuntimeError(
        f"this checkout predates the per-example provenance work: "
        f"jlens.generative is missing {missing}. Section 9 cannot verify "
        f"artifact association without it. Update the branch and rerun "
        f"section 2."
    )
print(f"provenance API present: {', '.join(REQUIRED_API)}")

import transformers

import torch as _torch

print(f"torch {_torch.__version__} / transformers {transformers.__version__}")

## 3. Hugging Face authentication

`google/gemma-4-E4B-it` is gated: accept the licence on the model page with the
same account first.

The token is read with `getpass` (never echoed), placed in `HF_TOKEN` for this
process only, and **never printed, logged, or written to disk**. Only its length
and a masked prefix are shown, so you can tell a paste succeeded without
exposing the value.

In [ ]:
# 3. HF token via getpass. Never printed, never persisted.
import getpass
import os

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN already set for this process — leaving it alone.")
else:
    _token = getpass.getpass("Hugging Face token (input hidden): ").strip()
    if not _token:
        raise RuntimeError("No token entered; the gated checkpoint cannot be fetched.")
    os.environ["HF_TOKEN"] = _token
    del _token

_value = os.environ["HF_TOKEN"]
print(f"HF_TOKEN set: {len(_value)} chars, starts {_value[:3]}{'*' * 8} (masked)")
del _value

# The runner reads HF_TOKEN from the environment; these keep other libraries
# from prompting again. Nothing is written to ~/.huggingface.
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ["HF_TOKEN"])
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")

## 4. Google Drive

Everything durable lives under `MyDrive/jacobian-lens-gemma/`:

| path | contents |
|---|---|
| `runs/` | run directories, written **directly** by the experiment |
| `runs/pilot_.../artifacts/lens.pt` | the frozen pilot lens (section 5) |
| `logs/` | full stdout+stderr of each run |
| `archives/` | zipped completed runs (section 10) |

Mounting is idempotent; rerunning this cell will not remount or clear anything.

In [ ]:
# 4. Mount Drive and resolve the persistent paths. Creates, never deletes.
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNT = Path("/content/drive")
    if not (DRIVE_MOUNT / "MyDrive").is_dir():
        raise RuntimeError(
            "Drive did not mount: /content/drive/MyDrive is missing. Approve the "
            "authorization prompt and rerun this cell."
        )
    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
else:
    PERSIST_ROOT = Path.cwd()
    print("Not in Colab — using the local checkout for persistence.")

RUNS_ROOT = PERSIST_ROOT / "runs"
LOGS_ROOT = PERSIST_ROOT / "logs"
ARCHIVE_ROOT = PERSIST_ROOT / "archives"
for directory in (RUNS_ROOT, LOGS_ROOT, ARCHIVE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print(f"PERSIST_ROOT = {PERSIST_ROOT}")
print(f"RUNS_ROOT    = {RUNS_ROOT}")
print(f"LOGS_ROOT    = {LOGS_ROOT}")
print(f"ARCHIVE_ROOT = {ARCHIVE_ROOT}")
existing = sorted(p.name for p in RUNS_ROOT.glob("generative_*"))
print(f"existing generative runs ({len(existing)}): {existing[-5:]}")

## 5. Lens restoration and checksum verification

The runner reads the lens from
`<runs-root>/pilot_20260715T200437612150_311fd108c23a/artifacts/lens.pt`, so it
must sit at that path under the Drive runs root.

The artifact is verified **before** anything else runs:

- size `91,753,066` bytes
- SHA-256 `7229c7562d1d55420b70abb13f481934649c4b01417bd851e97cedb47c96f474`

A mismatch aborts with a clear message. The lens is frozen; a differing file is
a different lens, and any result computed against it would be uninterpretable.

If `lens.pt` is not already on Drive, run the upload cell: it accepts either a
whole `lens.pt` or ordered shards named `lens.pt.part00`, `lens.pt.part01`, …
(Colab's uploader is unreliable above ~100 MB, hence the shard path.)

In [ ]:
# 5a. Locate / verify the frozen lens. Read-only unless a restore is needed.
import hashlib
import shutil

LENS_RUN_DIR_NAME = "pilot_20260715T200437612150_311fd108c23a"
LENS_RELPATH = "artifacts/lens.pt"
LENS_PATH = RUNS_ROOT / LENS_RUN_DIR_NAME / LENS_RELPATH
EXPECTED_LENS_BYTES = 91_753_066
EXPECTED_LENS_SHA256 = (
    "7229c7562d1d55420b70abb13f481934649c4b01417bd851e97cedb47c96f474"
)


def sha256_file(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def verify_lens(path):
    """(ok, message). Never modifies or deletes the file."""
    path = Path(path)
    if not path.is_file():
        return False, f"not found: {path}"
    size = path.stat().st_size
    if size != EXPECTED_LENS_BYTES:
        return False, (
            f"size mismatch: {size:,} bytes, expected {EXPECTED_LENS_BYTES:,}"
        )
    digest = sha256_file(path)
    if digest != EXPECTED_LENS_SHA256:
        return False, (
            f"SHA-256 mismatch:\n  actual   {digest}\n  expected {EXPECTED_LENS_SHA256}"
        )
    return True, f"size {size:,} bytes, sha256 {digest}"


LENS_PATH.parent.mkdir(parents=True, exist_ok=True)
LENS_OK, LENS_MESSAGE = verify_lens(LENS_PATH)
print(f"lens path: {LENS_PATH}")
print(("OK  " if LENS_OK else "NOT VERIFIED  ") + LENS_MESSAGE)
if not LENS_OK:
    print(
        "\nRun cell 5b to restore lens.pt (whole file or ordered shards), "
        "then rerun this cell. Nothing downstream may run until this verifies."
    )

In [ ]:
# 5b. Restore lens.pt from an upload. Only runs if 5a did not verify.
# Accepts a whole lens.pt, or shards named lens.pt.part00, lens.pt.part01, ...
# Writes to a temporary file and only moves it into place after verification,
# so a failed restore can never replace a good lens with a bad one.
if LENS_OK:
    print("Lens already verified — nothing to restore. (Cell is a no-op.)")
else:
    if not IN_COLAB:
        raise RuntimeError(
            f"Lens missing at {LENS_PATH} and uploads are Colab-only. Copy the "
            f"verified lens.pt to that path manually."
        )
    from google.colab import files

    print("Select lens.pt, or all lens.pt.partNN shards at once.")
    uploaded = files.upload()
    staging = LENS_PATH.parent / "lens.pt.incoming"
    names = sorted(uploaded)
    shards = [n for n in names if ".part" in n]
    if shards:
        print(f"reassembling {len(shards)} shards: {shards}")
        with open(staging, "wb") as out:
            for name in shards:
                out.write(uploaded[name])
    elif len(names) == 1:
        with open(staging, "wb") as out:
            out.write(uploaded[names[0]])
    else:
        raise RuntimeError(f"expected lens.pt or lens.pt.partNN shards, got {names}")

    ok, message = verify_lens(staging)
    if not ok:
        staging.unlink(missing_ok=True)
        raise RuntimeError(
            f"ABORT — restored lens failed verification and was discarded.\n"
            f"{message}\n"
            f"The pilot lens is frozen; a file with a different fingerprint is a "
            f"different lens, and any result computed against it is meaningless."
        )
    shutil.move(str(staging), str(LENS_PATH))
    LENS_OK, LENS_MESSAGE = verify_lens(LENS_PATH)
    print(f"restored and verified: {LENS_MESSAGE}")

In [ ]:
# 5c. Hard gate. Nothing below may run against an unverified lens.
if not LENS_OK:
    raise RuntimeError(
        f"ABORT — lens verification failed: {LENS_MESSAGE}\n"
        f"expected {EXPECTED_LENS_BYTES:,} bytes, sha256 {EXPECTED_LENS_SHA256}\n"
        f"at {LENS_PATH}"
    )
print(f"Lens verified. {LENS_MESSAGE}")

## 6. Target-token validation

Loads the **tokenizer only** (a few MB, no weights, no GPU) and checks that every
benchmark target segments to 2–6 tokens *contextually* — as the assistant
continuation of each formatted receiver prompt, which is how the run scores them.

A single-token target would make `prompt_only` / `constant` / `decaying`
produce identical target log-probabilities, so this is a genuine pre-flight gate,
not a formality. Cheap enough to always run before spending GPU time.

In [ ]:
# 6. Stream the target-token validator.
import subprocess
import sys

CONFIG_PATH = "configs/gemma_generative_validation.yaml"


def stream(argv, cwd=None, env=None):
    """Run a command, echoing stdout+stderr live. Returns (returncode, lines)."""
    process = subprocess.Popen(
        argv,
        cwd=str(cwd or CHECKOUT_DIR),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lines.append(line)
    process.wait()
    return process.returncode, lines


rc, _ = stream(
    [sys.executable, "-u", "scripts/validate_benchmark_targets.py",
     "--config", CONFIG_PATH],
    env={**os.environ},
)
print(f"\nreturn code: {rc}")
if rc != 0:
    raise RuntimeError(
        f"target-token validation failed ({rc}). Fix the benchmark before "
        f"spending GPU time; the run would abort on the same check anyway."
    )
print("TARGET TOKENIZATION OK")

## 7. The corrected two-concept experiment

```
python -u scripts/run_generative_validation.py \
    --config configs/gemma_generative_validation.yaml \
    --allow-model-load --device-map cuda \
    --runs-root <DRIVE>/runs \
    --limit-examples 2 --layers 14 21 --ratios 0.05 0.1 0.25
```

`--limit-examples 2` on the `dev` split runs `dev-phrase-solar-eclipse` and
`dev-entity-mandela`; each is the other's unrelated-cone donor. `held-phrase-black-hole`
is in the **heldout** split and cannot be reached — section 9 verifies that from
the records rather than assuming it.

The run writes **directly** to Drive, so an interrupted VM leaves every completed
record on disk (`records.jsonl` is fsynced per record).

Cell 7a launches the subprocess in the background; cell 7b streams it to
completion. Splitting them is what lets the section 8 monitor run against a live
process: interrupt 7b at any time and the experiment keeps going.

In [ ]:
# 7a. Launch the experiment in the background, teeing to a Drive-backed log.
import datetime
import subprocess
import sys
import threading
import time
from collections import deque
from pathlib import Path

LIMIT_EXAMPLES = 2
LAYERS = ["14", "21"]
RATIOS = ["0.05", "0.1", "0.25"]

EXPERIMENT_ARGV = [
    sys.executable, "-u", "scripts/run_generative_validation.py",
    "--config", CONFIG_PATH,
    "--allow-model-load",
    "--device-map", "cuda",
    "--runs-root", str(RUNS_ROOT),
    "--limit-examples", str(LIMIT_EXAMPLES),
    "--layers", *LAYERS,
    "--ratios", *RATIOS,
]

EXPERIMENT = globals().get("EXPERIMENT")
if EXPERIMENT is not None and EXPERIMENT["process"].poll() is None:
    print(
        f"An experiment is already running (pid {EXPERIMENT['process'].pid}, "
        f"started {EXPERIMENT['started_iso']}).\n"
        f"Log: {EXPERIMENT['log_path']}\n"
        f"Run cell 7b to stream it, or section 8 to monitor. Not launching "
        f"a second one."
    )
else:
    _stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    LOG_PATH = LOGS_ROOT / f"generative_run_{_stamp}.log"
    _handle = open(LOG_PATH, "a", encoding="utf-8", buffering=1)
    _handle.write(f"# argv: {' '.join(EXPERIMENT_ARGV)}\n")
    _handle.write(f"# started_utc: {_stamp}\n")
    _handle.write(f"# repo HEAD: {HEAD}\n")

    _process = subprocess.Popen(
        EXPERIMENT_ARGV,
        cwd=str(CHECKOUT_DIR),
        env={**os.environ},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    _tail = deque(maxlen=400)

    def _pump(process=_process, handle=_handle, tail=_tail):
        for line in process.stdout:
            handle.write(line)
            tail.append(line)
        process.wait()
        handle.write(f"# return_code: {process.returncode}\n")
        handle.close()

    _thread = threading.Thread(target=_pump, daemon=True)
    _thread.start()

    EXPERIMENT = {
        "process": _process,
        "thread": _thread,
        "log_path": LOG_PATH,
        "tail": _tail,
        "started": time.time(),
        "started_iso": _stamp,
        "argv": EXPERIMENT_ARGV,
    }
    print(f"launched pid {_process.pid}")
    print(f"argv: {' '.join(EXPERIMENT_ARGV)}")
    print(f"log : {LOG_PATH}")
    print("\nRun cell 7b to stream it live.")

In [ ]:
# 7b. Stream the running experiment to completion; report elapsed, return code,
# and a success/failure banner. On failure, print the last 100 log lines.
# Interrupting this cell does NOT stop the experiment — rerun it to reattach.
import time

if EXPERIMENT is None:
    raise RuntimeError("No experiment launched. Run cell 7a first.")

_log = EXPERIMENT["log_path"]
_position = 0
_interrupted = False
try:
    while True:
        with open(_log, encoding="utf-8", errors="replace") as handle:
            handle.seek(_position)
            chunk = handle.read()
            _position = handle.tell()
        if chunk:
            sys.stdout.write(chunk)
            sys.stdout.flush()
        if EXPERIMENT["process"].poll() is not None and not chunk:
            break
        time.sleep(1.0)
except KeyboardInterrupt:
    _interrupted = True

elapsed = time.time() - EXPERIMENT["started"]
rc = EXPERIMENT["process"].poll()
print("\n" + "=" * 72)
print(f"elapsed     : {elapsed / 60:.1f} min ({elapsed:.0f} s)")
print(f"return code : {rc}")
print(f"log         : {_log}")
if _interrupted and rc is None:
    print("=" * 72)
    print("STREAM DETACHED — the experiment is STILL RUNNING.")
    print("Rerun this cell to reattach, or use section 8 to monitor.")
elif rc == 0:
    print("=" * 72)
    print("  SUCCESS — run completed. Continue to section 9.")
    print("=" * 72)
else:
    print("=" * 72)
    print(f"  FAILURE — the experiment exited with code {rc}.")
    print("  Last 100 log lines:")
    print("=" * 72)
    with open(_log, encoding="utf-8", errors="replace") as handle:
        for line in handle.read().splitlines()[-100:]:
            print(line)
    print("=" * 72)
    print(f"  Full log: {_log}")
    print("=" * 72)

## 8. Live monitoring

A **stoppable** loop showing GPU state, process status, the latest log lines, and
how many records have landed so far. Stop it with the interrupt button
(`KeyboardInterrupt`) or let it hit `MONITOR_MAX_SECONDS`; either way the
experiment keeps running.

Safe to run at any time — including in place of cell 7b, or after reconnecting
to a runtime whose 7b output was lost.

In [ ]:
# 8. Stoppable monitor: nvidia-smi, GPU memory/utilization, process status,
# the last 20 log lines, and the current records.jsonl line count.
import shutil
import subprocess
import time
from pathlib import Path

MONITOR_INTERVAL_SECONDS = 20
MONITOR_MAX_SECONDS = 3600


def _gpu_line():
    if not shutil.which("nvidia-smi"):
        return "nvidia-smi unavailable"
    query = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu",
         "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    return query.stdout.strip() or query.stderr.strip()


def _newest_run_dir():
    candidates = sorted(
        (p for p in RUNS_ROOT.glob("generative_*") if p.is_dir()),
        key=lambda p: p.name,
    )
    return candidates[-1] if candidates else None


def _record_count(run_dir):
    if run_dir is None:
        return None, None
    records = run_dir / "artifacts" / "records.jsonl"
    if not records.is_file():
        return records, 0
    with open(records, encoding="utf-8", errors="replace") as handle:
        return records, sum(1 for line in handle if line.strip())


def monitor(interval=MONITOR_INTERVAL_SECONDS, max_seconds=MONITOR_MAX_SECONDS):
    started = time.time()
    log_path = EXPERIMENT["log_path"] if EXPERIMENT else None
    try:
        while time.time() - started < max_seconds:
            now = time.strftime("%H:%M:%S")
            if EXPERIMENT is None:
                status = "no experiment launched in this session"
            else:
                rc = EXPERIMENT["process"].poll()
                age = (time.time() - EXPERIMENT["started"]) / 60
                status = (
                    f"pid {EXPERIMENT['process'].pid} RUNNING ({age:.1f} min)"
                    if rc is None
                    else f"pid {EXPERIMENT['process'].pid} EXITED rc={rc} ({age:.1f} min)"
                )
            run_dir = _newest_run_dir()
            records_path, n_records = _record_count(run_dir)

            print("=" * 72)
            print(f"[{now}] {status}")
            print(f"GPU     : {_gpu_line()}")
            print(f"run dir : {run_dir.name if run_dir else '(none yet)'}")
            print(f"records : {n_records if n_records is not None else '-'}"
                  f"  ({records_path if records_path else 'n/a'})")
            if log_path and Path(log_path).is_file():
                print(f"--- last 20 lines of {Path(log_path).name} ---")
                with open(log_path, encoding="utf-8", errors="replace") as handle:
                    for line in handle.read().splitlines()[-20:]:
                        print("  " + line)
            if EXPERIMENT is not None and EXPERIMENT["process"].poll() is not None:
                print("Process finished — stopping monitor.")
                return
            time.sleep(interval)
        print(f"Monitor stopped after MONITOR_MAX_SECONDS={max_seconds}s "
              f"(the experiment is unaffected).")
    except KeyboardInterrupt:
        print("\nMonitor stopped by interrupt. The experiment is unaffected.")


monitor()

## 9. Result and provenance inspection

Locates the newest run, verifies the expected artifacts, and prints:

- the record count,
- a compact **generated-output** table (what the model actually said),
- a **provenance** table of every identity / donor / activation / cone field
  that is present — missing fields are shown as `-`, never invented,
- an explicit **artifact-association check** for `dev-entity-mandela`.

The Mandela check compares each record's `source_activation_sha256` and
`cone_source_example_id` against the Mandela example's own, and flags any record
whose source or cone provenance differs. The unrelated-cone controls
(`unrelated_cone`, `natural_unrelated_cone`) *are supposed* to carry the donor's
cone, so they are reported separately as expected-by-design rather than as
anomalies.

In [ ]:
# 9a. Locate the newest run and verify its artifacts.
import json
from pathlib import Path

candidates = sorted(
    (p for p in RUNS_ROOT.glob("generative_*") if p.is_dir()), key=lambda p: p.name
)
if not candidates:
    raise RuntimeError(f"no generative_* run directory under {RUNS_ROOT}")
RUN_DIR = candidates[-1]
print(f"newest run: {RUN_DIR}")

EXPECTED_ARTIFACTS = {
    "artifacts/records.jsonl": True,
    "artifacts/gates.json": True,
    "artifacts/prompt_debug.json": True,
    "run_metadata.json": True,
    "summary.md": True,
    "artifacts/pursuits.json": False,
    "artifacts/targets.json": False,
    "artifacts/summary_by_condition.json": False,
    "artifacts/calibration.json": False,
    "artifacts/gonogo.json": False,
    "artifacts/natural_scale_comparison.json": False,
}
missing_required = []
for relpath, required in EXPECTED_ARTIFACTS.items():
    path = RUN_DIR / relpath
    mark = "OK " if path.is_file() else ("MISSING" if required else "absent ")
    size = f"{path.stat().st_size:,} B" if path.is_file() else "-"
    print(f"  {mark:8} {relpath:42} {size:>14}")
    if required and not path.is_file():
        missing_required.append(relpath)
if missing_required:
    print(f"\nWARNING: required artifacts missing: {missing_required}")
    print("The run did not complete; the tables below cover whatever landed.")

RECORDS_PATH = RUN_DIR / "artifacts" / "records.jsonl"
RECORDS = []
if RECORDS_PATH.is_file():
    with open(RECORDS_PATH, encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                RECORDS.append(json.loads(line))
print(f"\nrecords: {len(RECORDS)}")
if RECORDS:
    print(f"examples   : {sorted({r['example_id'] for r in RECORDS})}")
    print(f"layers     : {sorted({r['source_layer'] for r in RECORDS})}")
    print(f"conditions : {len(sorted({r['vector_condition'] for r in RECORDS}))}")
    print(f"prompts    : {sorted({r.get('neutral_prompt_id') for r in RECORDS})}")

summary_path = RUN_DIR / "summary.md"
if summary_path.is_file():
    print("\n--- summary.md ---")
    print(summary_path.read_text(encoding="utf-8"))

In [ ]:
# 9b. Compact table of what the model actually generated.
def table(rows, headers):
    """Fixed-width text table. Missing values print as '-', never invented."""
    if not rows:
        print("(no rows)")
        return
    cells = [[("-" if v is None else str(v)) for v in row] for row in rows]
    widths = [
        max(len(headers[i]), max(len(row[i]) for row in cells))
        for i in range(len(headers))
    ]
    print("  ".join(h.ljust(widths[i]) for i, h in enumerate(headers)))
    print("  ".join("-" * widths[i] for i in range(len(headers))))
    for row in cells:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(row))))


decoded = [r for r in RECORDS if r.get("generated_text") is not None]
print(f"decoded records: {len(decoded)} / {len(RECORDS)}\n")

rows = []
for r in sorted(
    decoded,
    key=lambda r: (
        r["example_id"],
        r["source_layer"],
        r.get("neutral_prompt_id") or "",
        r["vector_condition"],
        r.get("requested_ratio") if r.get("requested_ratio") is not None else -1.0,
    ),
):
    rows.append([
        r["example_id"],
        r["source_layer"],
        (r.get("neutral_prompt_id") or "")[:18],
        r["vector_condition"],
        r.get("requested_ratio"),
        (r.get("steering_schedule") or {}).get("kind"),
        repr(r["generated_text"])[:44],
        r.get("target_recovered_substring"),
    ])
table(rows, ["example", "L", "prompt", "condition", "ratio", "sched",
             "generated", "substr"])

# The unsteered baseline in isolation. The receiver prompt carries no example
# information, so `none` decodes are a property of the prompt alone: if the same
# string appears here for BOTH examples, a repeated output elsewhere is the
# model answering the prompt, not evidence of a swapped artifact.
print("\n--- unsteered baseline (`none`) by prompt ---")
baseline = {}
for r in decoded:
    if r["vector_condition"] == "none":
        baseline.setdefault(r.get("neutral_prompt_id"), {})[r["example_id"]] = (
            r["generated_text"]
        )
for prompt_id, by_example in sorted(baseline.items(), key=lambda kv: str(kv[0])):
    print(f"\n  prompt {prompt_id!r}")
    for example_id, text in sorted(by_example.items()):
        print(f"    {example_id:28} -> {text!r}")
    distinct = set(by_example.values())
    print(f"    distinct baseline outputs across examples: {len(distinct)}"
          + ("  (prompt-determined, as expected)" if len(distinct) == 1 else ""))

In [ ]:
# 9c. Provenance table: every identity / donor / activation / cone field that is
# actually present. Absent fields print as '-' and are never back-filled.
PROVENANCE_FIELDS = [
    ("example_id", "example"),
    ("source_example_id", "src_example"),
    ("cone_source_example_id", "cone_src"),
    ("donor_example_id", "donor"),
    ("source_layer", "L"),
    ("source_activation_norm", "act_norm"),
    ("source_activation_sha256", "act_sha"),
    ("cone_norm", "cone_norm"),
    ("cone_sha256", "cone_sha"),
    ("injected_delta_sha256", "delta_sha"),
    ("source_prompt", "source_prompt"),
    ("target_phrase", "target"),
]

present = set()
for r in RECORDS:
    present |= {k for k, v in (r.get("provenance") or {}).items() if v is not None}
available = [(key, label) for key, label in PROVENANCE_FIELDS if key in present]
absent = [key for key, _ in PROVENANCE_FIELDS if key not in present]
print(f"provenance fields present: {[k for k, _ in available]}")
if absent:
    print(f"provenance fields absent (shown as '-'): {absent}")
    print("These runs predate the provenance work, or the field does not apply "
          "to any recorded condition. Nothing is inferred for them.")


def short(value, n=10):
    if value is None:
        return None
    text = str(value)
    if text.startswith("sha256:"):
        return text[7 : 7 + n]
    return text if len(text) <= 34 else text[:31] + "..."


print()
seen = set()
rows = []
for r in sorted(RECORDS, key=lambda r: (r["example_id"], r["source_layer"],
                                        r["vector_condition"])):
    provenance = r.get("provenance") or {}
    key = (r["example_id"], r["source_layer"], r["vector_condition"])
    if key in seen:
        continue  # one row per (example, layer, condition); ratios repeat it
    seen.add(key)
    rows.append(
        [r["vector_condition"]]
        + [short(provenance.get(field)) for field, _ in available]
    )
table(rows, ["condition"] + [label for _, label in available])

In [ ]:
# 9d. Artifact-association check for dev-entity-mandela.
#
# Flags any Mandela record whose SOURCE provenance is not Mandela's own, or
# whose CONE provenance points at a different example outside the two
# unrelated-cone controls (where a donor cone is the control's whole point).
from jlens.generative import (
    CONE_SOURCE_DONOR_CONDITIONS,
    expected_cone_source_example_id,
)

TARGET_EXAMPLE = "dev-entity-mandela"
mandela = [r for r in RECORDS if r["example_id"] == TARGET_EXAMPLE]
print(f"{TARGET_EXAMPLE}: {len(mandela)} records")

if not mandela:
    print("No records for that example in this run — check skipped, nothing inferred.")
else:
    own_activation = {}
    for r in mandela:
        digest = (r.get("provenance") or {}).get("source_activation_sha256")
        if digest is not None:
            own_activation.setdefault(r["source_layer"], set()).add(digest)
    for layer, digests in sorted(own_activation.items()):
        print(f"  layer {layer}: {len(digests)} distinct source-activation "
              f"fingerprint(s) {[d[7:17] for d in sorted(digests)]}")

    anomalies, by_design, unverifiable = [], [], []
    for r in mandela:
        provenance = r.get("provenance") or {}
        condition = r["vector_condition"]
        source_id = provenance.get("source_example_id")
        cone_id = provenance.get("cone_source_example_id")
        donor_id = provenance.get("donor_example_id")
        if "source_example_id" not in provenance:
            unverifiable.append(r)
            continue
        if source_id != TARGET_EXAMPLE:
            anomalies.append((r, f"source_example_id={source_id!r}"))
            continue
        digests = own_activation.get(r["source_layer"], set())
        if len(digests) > 1:
            anomalies.append((r, f"{len(digests)} activation fingerprints at "
                                 f"layer {r['source_layer']}"))
            continue
        try:
            expected = expected_cone_source_example_id(
                condition, example_id=TARGET_EXAMPLE, donor_example_id=donor_id
            )
        except Exception as exc:
            anomalies.append((r, f"cone role unresolvable: {exc}"))
            continue
        if cone_id != expected:
            anomalies.append((r, f"cone_source_example_id={cone_id!r}, "
                                 f"expected {expected!r}"))
        elif condition in CONE_SOURCE_DONOR_CONDITIONS:
            by_design.append(r)

    print(f"\n  anomalies                    : {len(anomalies)}")
    print(f"  donor cone by design         : {len(by_design)} "
          f"({', '.join(sorted(CONE_SOURCE_DONOR_CONDITIONS))})")
    print(f"  unverifiable (no provenance) : {len(unverifiable)}")

    if anomalies:
        print("\n  FLAGGED — provenance differs from the Mandela example:")
        for r, reason in anomalies[:40]:
            print(f"    L{r['source_layer']} {r['vector_condition']:26} "
                  f"ratio={r.get('requested_ratio')} :: {reason}")
        if len(anomalies) > 40:
            print(f"    ... and {len(anomalies) - 40} more")
    elif unverifiable:
        print("\n  NOT VERIFIABLE — these records carry no identity block.")
    else:
        print("\n  CLEAN — every Mandela record used Mandela's own source "
              "activation, and the only records carrying another example's cone "
              "are the unrelated-cone controls, which is their purpose.")

    # Cross-example distinctness: the check above verifies that every Mandela
    # record agrees with itself. This verifies that "Mandela's activation" is
    # actually distinguishable from the other example's — otherwise agreement
    # would be trivially satisfiable by every example sharing one activation.
    print("\n  source-activation fingerprints across examples:")
    for layer in sorted({r["source_layer"] for r in RECORDS}):
        per_example = {}
        for r in RECORDS:
            if r["source_layer"] != layer:
                continue
            digest = (r.get("provenance") or {}).get("source_activation_sha256")
            if digest is not None:
                per_example.setdefault(r["example_id"], set()).add(digest)
        flat = {next(iter(v)) for v in per_example.values() if len(v) == 1}
        verdict = (
            "distinct" if len(flat) == len(per_example) else "COLLIDING"
        )
        print(f"    layer {layer}: {len(per_example)} example(s), "
              f"{len(flat)} distinct fingerprint(s) -> {verdict}")
        for example_id, digests in sorted(per_example.items()):
            print(f"      {example_id:28} {sorted(d[7:17] for d in digests)}")

    donors = {(r.get("provenance") or {}).get("donor_example_id") for r in mandela}
    print(f"\n  unrelated-cone donor(s) for {TARGET_EXAMPLE}: "
          f"{sorted(d for d in donors if d)}")
    heldout_seen = sorted(
        d for d in donors if d and str(d).startswith("held-")
    )
    print(f"  held-out donors present: {heldout_seen or 'none'}")

    # Any generated text that names another benchmark target, listed with its
    # provenance so the two explanations are distinguishable at a glance.
    other_targets = {r["target_phrase"].strip().lower() for r in RECORDS} - {
        r["target_phrase"].strip().lower() for r in mandela
    }
    other_targets |= {"black hole"}
    hits = [
        r for r in mandela
        if r.get("generated_text")
        and any(t in r["generated_text"].lower() for t in other_targets if t)
    ]
    print(f"\n  Mandela decodes containing another benchmark target's surface "
          f"form: {len(hits)}")
    for r in hits[:20]:
        provenance = r.get("provenance") or {}
        print(f"    L{r['source_layer']} {r['vector_condition']:24} "
              f"ratio={str(r.get('requested_ratio')):>5} "
              f"cone_src={provenance.get('cone_source_example_id')} "
              f"-> {r['generated_text']!r}")
    if hits:
        print("\n  Read this against the `none` baseline block in 9b: if the "
              "unsteered decode for BOTH examples is the same string, these are "
              "the model answering the receiver prompt, not a swapped artifact. "
              "The provenance verdict above is what settles it either way.")

## 10. Persistent archive and export

Zips the completed run directory together with its log into
`MyDrive/jacobian-lens-gemma/archives/`, computes the archive's SHA-256, and
prints the persistent paths.

Previous runs and archives are never deleted. Re-archiving an existing run
rewrites only that run's own archive.

In [ ]:
# 10. Archive the run + log to Drive, fingerprint it, and print the paths.
import zipfile

ARCHIVE_PATH = ARCHIVE_ROOT / f"{RUN_DIR.name}.zip"
LOG_FOR_RUN = EXPERIMENT["log_path"] if EXPERIMENT else None

n_files = 0
with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path(RUN_DIR.name) / path.relative_to(RUN_DIR)))
            n_files += 1
    if LOG_FOR_RUN and Path(LOG_FOR_RUN).is_file():
        archive.write(LOG_FOR_RUN, arcname=str(Path(RUN_DIR.name) / "logs" / Path(LOG_FOR_RUN).name))
        n_files += 1

archive_sha = sha256_file(ARCHIVE_PATH)
print(f"archived {n_files} file(s)")
print()
print("PERSISTENT PATHS (survive VM termination):")
print(f"  run directory : {RUN_DIR}")
print(f"  log           : {LOG_FOR_RUN or '(no log from this session)'}")
print(f"  archive       : {ARCHIVE_PATH}")
print(f"  archive size  : {ARCHIVE_PATH.stat().st_size:,} bytes")
print(f"  archive sha256: {archive_sha}")
print()
print("All archives on Drive:")
for path in sorted(ARCHIVE_ROOT.glob("*.zip")):
    print(f"  {path.name:56} {path.stat().st_size:>12,} B")